In [2]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub


In [3]:
from langchain.agents import create_react_agent, AgentExecutor

#create_react_agent = reasioning and action agent

In [6]:
#LOAD ENV VARIABLES

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
#INETRNET SEARCH RESULTS
search_tool = TavilySearchResults(max_results=2)

In [8]:
result = search_tool.invoke("give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Retail & Logistics AI\n\nJune 19, 2026\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026\n\n### Google folds Display Ads into AI-first Demand Gen platform\n\nMarketing AI\n\nMay 27, 2026\n\nImage 25\n\n#### Enterprise\n\n### Anthropic deploys Claude Sonnet 5, Fable and Mythos restored\n\nInside AI\n\nJuly 1, 2026\n\n### HP accelerates enterprise workflows with OpenAI Frontier\n\nWorld of Work\n\nJune 29, 2026\n\n### Insurers pivot AI strategy toward core risk underwriting\n\nWorld of Work\n\nJune 16, 2026\n\n#### Industries\n\n### HP accelerates enterprise workflows with OpenAI Frontier\n\nWorld of Work\n\nJune 29, 2026\n\n### Omio scales travel product development using OpenAI models\n\nAI in Action\n\nJune 23, 2026\n\n### L’Oréal brings Maybelline virtual try-on to ChatGPT [...] #### Applications\n\n### Thailand becomes one of the first in Asia to get the Sora ap

In [11]:
#LLM

from langchain_groq import ChatGroq

llm = ChatGroq(

model_name="llama-3.1-8b-instant",

temperature=0

)



In [12]:
response = llm.invoke("what year is it?")
response

AIMessage(content='It is currently 2023.', response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 40, 'total_tokens': 48, 'completion_time': 0.020967194, 'completion_tokens_details': None, 'prompt_time': 0.00196393, 'prompt_tokens_details': None, 'queue_time': 0.04676937, 'total_time': 0.022931124}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-f3db44ba-2040-4379-8c5d-9d7620f26229-0')

In [13]:
#PROMPT (according to react agent we have rpedefined rpompt here in langchain hub)

prompt = hub.pull("hwchase17/react")


c:\Users\santh\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [16]:
#TOOLS


tools = [search_tool]

In [17]:
#CREATE AGENT

agent = create_react_agent(
    llm = llm,
    tools = tools,
    prompt = prompt
)

In [ ]:
#WE HAVE TO RUN THE AGENT WITH THE HELP OF AGENT EXECUTOR
#EXECUTOR

agent_executor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose = True
)

#verbose = True we can see logs whatever thought, action and observation your agent is doing


In [ ]:
#RUN

response = agent_executor.invoke({
    "input": (
        "Find the capital of india"
        "and find its current weather."
    )
})

In [21]:
print(response["output"])

The capital of India is New Delhi, and the current weather in New Delhi is 35°C with a feels-like temperature of 36°C, and there is a 36% chance of rain. The 10-day forecast shows a trend of severe sandstorms and haze conditions.


In [ ]:
#whenever you dont give any tools, it runs tools the loop will be finished after exhausting iteration limit or time limit

In [ ]:
#llm is used for reasoning and tool for latest info when i/p prompt comes to agent to rpoduce a output


In [ ]:
#ReAct is a design pattern used in AI agents that stands for Reasoning and Acting 
#it allows LLM to interleave internal reasoning (Thought) with external actions (like tool use) in a structured multi step process 
# #instead of generating answer in one go , the model thinks step by step, deciding what it shoudl do next adn optionally calling api tools to help it




what is ReAct design pattern ?

-> traditionally LLM's operate in one shot generation mode- you give the prompt and they try to guess the entire answer immediately
-> the ReAct design pattern introduced in 2022 paper chnages this by forcing LLM to alternate between Reasoning("Thinking") and Acting("Executing a tool")
-> instead of guesssing the model writes out its thought process ,realizes it needs information it dosent have, then calls tools, looks at result and repeats until it fugure outs the answer

Decounstructing the template:

text you see ins a structured instructions template injected into the LLM before your question.


The text you see on your screen is a structured instructions template injected into the LLM before your question. Here is what each piece means:

{tools}: A placeholder where the developer injects a description of available tools (e.g., "Search: A web search engine. Useful for answering questions about current events.").

Question: The actual query the user inputs.

Thought: A mandatory step where the LLM talks to itself. It analyzes the question and determines what to do next.

Action: The specific tool the LLM decides to use (must be one of the {tool_names}).

Action Input: The parameters or query the LLM passes to that tool (e.g., the exact search query).

Observation: This is crucial. The LLM does not write this part. The application framework (like LangChain) runs the tool, grabs the result, and types it back into the prompt history for the LLM to see.

Final Answer: The ultimate conclusion the LLM reaches once it has gathered enough observations.

How the LLM and the Application Work Hand-in-Hand
An LLM on its own cannot browse the web, run python scripts, or call banking APIs. It is just a text predictor. The ReAct pattern works like a game of ping-pong between the LLM and your Application Code (the Agent Framework).

Here is the step-by-step lifecycle of a single request:

Step A: The Setup
Your application sends the prompt template from your screen to the LLM, filling in the {tools} it has access to and the user's {input} question.

Step B: The LLM Reasons & Acts
The LLM reads everything and begins generating text. Because of the template's strict instructions, it outputs:

Thought: I need to find the current price of stock X, but my training data is cut off. I should use the Search tool.
Action: Search
Action Input: "Stock price of X today"

Step C: The Interception (The App Takes Over)
The application code watches the LLM's output. The moment it sees the text stop after Action Input: "...", the application pauses the LLM.
The code parses the text, extracts the tool name (Search) and the input (Stock price of X today), and executes that actual code block or API call.

Step D: The Feed-back Loop
The application gets the result from the tool (e.g., "$150"). It appends that text directly to the prompt history under a new line:

Observation: $150

Step E: The Final Evaluation
The application hands the updated text history back to the LLM and says, "Keep generating." The LLM reads its own past thoughts, sees the new Observation, and realizes it has the answer:

Thought: I now know the final answer based on the observation.
Final Answer: The current stock price of X is $150.

By breaking the problem down into explicit steps the ReAct pattern dramatically reduces LLM hallucinations. 
It forces the model to ground its final answers in real-world Observations rather than relying purely on static training data weights.

<h1>AGENT and AGENT EXECUTOR</h1>


-> internally agent does 3 things

. thought
. action
. observation

without agent executor we cant run these 3 steps

agent execuotr orchestrates the entire loop:
> sends input and previous messages(thought, action ,observation)to agent
agent takes everything gives it a thought and then thought of taking a action(caliing a tool), so it reaches out to executor
> gets the next action from the agent
> executes the tool with the provided input
> adds the tool observation(T,A,O) back into history
> loops again until the updated history until agent says final answer




if you want to create your own tool(custom tool) you can do it by importing tool from langchain.tools
and use it as a decerator above the function you are gonan call
we can use that function name in tools to use it in calling